## 🔧 Paso 1: Instalación de dependencias
Instalamos todas las bibliotecas necesarias para trabajar con LangChain, embeddings, FAISS, y MLflow en Databricks.

In [0]:
%pip install -U --quiet \
  databricks-sdk==0.49.0 \
  "databricks-langchain>=0.4.0" \
  databricks-agents \
  "mlflow[databricks,langchain]==2.22.0" \
  langchain==0.3.19 \
  langchain_core==0.3.37 \
  langchain-community==0.3.7 \
  pydantic==2.10.1 \
  bs4==0.0.2 \
  markdownify==0.14.1 \
  youtube_search \
  Wikipedia \
  grandalf \
  pypdf \
  faiss-cpu

dbutils.library.restartPython()

## 📦 Paso 2: Importación de librerías

In [0]:
# Standard libraries
import os
import requests
from pathlib import Path
import uuid
import tempfile

# Spark libraries
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

# MLflow library
import mlflow

# Langchain libraries
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chat_models import ChatDatabricks
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableLambda
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import DatabricksEmbeddings

# Databricks Serving Endpoint
from databricks import agents
from mlflow.models.resources import DatabricksServingEndpoint

# Utility
from operator import itemgetter

## 📥 Paso 2: Descarga de documentos fuente
Descargamos los PDFs desde un repositorio público (GitHub) y los almacenamos temporalmente en `/tmp` para procesarlos.

In [0]:
# Ruta del repositorio
base_url = "https://raw.githubusercontent.com/darkanita/GenAIOps_Pycon2025/main/data/pdfs"
pdf_names = [
    "Benefit_Options.pdf",
    "Northwind_Health_Plus_Benefits_Details.pdf",
    "Northwind_Standard_Benefits_Details.pdf",
    "PerksPlus.pdf",
    "employee_handbook.pdf",
    "role_library.pdf"
]

local_dir = "/tmp/genaiops_pdfs"
Path(local_dir).mkdir(parents=True, exist_ok=True)

# Descargar los archivos
for name in pdf_names:
    url = f"{base_url}/{name}"
    response = requests.get(url)
    if response.status_code == 200:
        with open(os.path.join(local_dir, name), "wb") as f:
            f.write(response.content)
        print(f"✅ Descargado: {name}")
    else:
        print(f"❌ Error al descargar: {name}")


In [0]:
# Cargar documentos PDF desde el directorio local y extraer su contenido
documents = []
for file in Path(local_dir).glob("*.pdf"):
    loader = PyPDFLoader(str(file))
    documents.extend(loader.load())

## ✂️ Paso 3: Carga y partición de documentos (Chunking)
Cargamos los PDFs como documentos y los separamos en fragmentos de texto (chunks) utilizando una estrategia definida de tamaño y solapamiento.

In [0]:
# Definimos y registramos la estrategia
chunk_size = 500
chunk_overlap = 50
chunking_strategy = f"rctxt-{chunk_size}-{chunk_overlap}"

with mlflow.start_run(run_name="chunking_strategy_v1"):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs_chunked = text_splitter.split_documents(documents)

    # Track en MLflow
    mlflow.log_param("chunk_size", chunk_size)  # Registra el tamaño de los chunks
    mlflow.log_param("chunk_overlap", chunk_overlap)  # Registra el solapamiento de los chunks
    mlflow.log_param("strategy_name", chunking_strategy)  # Registra el nombre de la estrategia
    mlflow.log_metric("n_chunks", len(docs_chunked))  # Registra el número de chunks generados

In [0]:
# Convertir los chunks a filas
data = []
for i, doc in enumerate(docs_chunked):
    data.append({
        "id": str(uuid.uuid4()),  # Generar un UUID único para cada chunk
        "text": doc.page_content,  # Contenido del chunk
        "source": doc.metadata.get("source", "unknown")  # Fuente del documento original
    })

# Crear un DataFrame Spark con el esquema especificado
schema = StructType([
    StructField("id", StringType(), False),  # Campo 'id' de tipo String, no nulo
    StructField("text", StringType(), False),  # Campo 'text' de tipo String, no nulo
    StructField("source", StringType(), True)  # Campo 'source' de tipo String, puede ser nulo
])

# Crear un DataFrame Spark con los datos y el esquema especificado
df = spark.createDataFrame(data, schema)

In [0]:
# Nombre de la tabla donde se guardarán los chunks
table_name = "genaiops_chunks"

# Guardar el DataFrame en una tabla Delta, sobrescribiendo si ya existe
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
# Visualizar los primeros 5 registros de la tabla Delta
df = spark.table("genaiops_chunks")
display(df.limit(5))

## 🧠 Paso 4: Generación de Embeddings y creación del índice FAISS
Utilizamos un modelo de embeddings de Databricks para transformar los chunks en vectores y los almacenamos en FAISS, una base de datos vectorial local.

In [0]:
# Crear un modelo de embeddings utilizando el endpoint especificado
embedding_model = DatabricksEmbeddings(endpoint="databricks-gte-large-en")

In [0]:
# Crear un índice FAISS a partir de los documentos chunked utilizando el modelo de embeddings
faiss_index = FAISS.from_documents(docs_chunked, embedding_model)

# Definir la ruta donde se guardará el índice FAISS
faiss_path = "/tmp/genaiops_faiss"

# Guardar el índice FAISS en disco
faiss_index.save_local(faiss_path)

In [0]:
# Cargar el índice FAISS desde el disco
faiss_loaded = FAISS.load_local(faiss_path, embeddings=embedding_model, allow_dangerous_deserialization=True)

# Obtener un registro de la base de datos vectorial
registro = faiss_loaded.index.reconstruct(0)

# Mostrar el registro
display(registro)

In [0]:
# Convertir el índice FAISS cargado en un recuperador
retriever = faiss_loaded.as_retriever()

# Definir la pregunta para la búsqueda de similitud
question = "¿Qué cubre el plan de beneficios Northwind Health Plus?"

# Realizar una búsqueda de similitud utilizando el índice FAISS
docs = faiss_index.similarity_search(query=question, k=5)

# Mostrar el contenido de los documentos recuperados
for doc in docs:
    print("📄 Texto completo:\n", doc.page_content)
    print("📎 Metadata:", doc.metadata)

## 🤖 Paso 5: Definición de la RAG Chain
Creamos una función `create_chain(model_config)` que construye todo el pipeline RAG, incluyendo recuperación desde FAISS, formateo del contexto y paso por el LLM.

In [0]:
# Configuración de la cadena
chain_config = {
    "llm_model_serving_endpoint_name": "databricks-meta-llama-3-3-70b-instruct",  # el modelo base que queremos usar
    "embedding_model": "databricks-gte-large-en",  # Modelo de Embedding
    "llm_prompt_template": """You are an assistant that answers questions. Use the following pieces of retrieved context to answer the question. Some pieces of context may be irrelevant, in which case you should not use them to form the answer.\n\nContext: {context}""",
}

## Enable MLflow Tracing
mlflow.langchain.autolog()

# Configuración del modelo
model_config = mlflow.models.ModelConfig(development_config=chain_config)

# Method to format the docs returned by the retriever into the prompt (keep only the text from chunks)
def format_context(docs):
    chunk_contents = [f"Passage: {d.page_content}\n" for d in docs]
    return "".join(chunk_contents)

#Let's try our retriever chain:
relevant_docs = (retriever | RunnableLambda(format_context)| StrOutputParser()).invoke("¿Qué cubre el plan de beneficios Northwind Health Plus?")

print(relevant_docs)

In [0]:
# Plantilla de prompt para el chat
prompt = ChatPromptTemplate.from_messages(
    [  
        ("system", model_config.get("llm_prompt_template")), # Contiene las instrucciones de la configuración
        ("user", "{question}") # Preguntas del usuario
    ]
)

# Nuestro modelo base respondiendo al prompt final
model = ChatDatabricks(
    endpoint=model_config.get("llm_model_serving_endpoint_name"),
    extra_params={"temperature": 0.01, "max_tokens": 500}
)

# Probar nuestro prompt
answer = (prompt | model | StrOutputParser()).invoke({'question':"¿Qué cubre el plan de beneficios Northwind Health Plus?", 'context': ''})
print(answer)

In [0]:
# Devuelve el contenido de la cadena más reciente de mensajes: [{...}] del usuario para ser usado como pregunta de entrada
def extract_user_query_string(chat_messages_array):
    return chat_messages_array[-1]["content"]

# Configuración del recuperador FAISS
faiss_retriever = faiss_index.as_retriever(search_kwargs={"k": 3})

# Cadena RAG (Recuperación-Augmentación-Generación)
chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query_string),  # Extrae la pregunta del usuario
        "context": itemgetter("messages")
        | RunnableLambda(extract_user_query_string)
        | faiss_retriever  # Recupera los documentos relevantes usando FAISS
        | RunnableLambda(format_context),  # Formatea los documentos recuperados
    }
    | prompt  # Genera el prompt final
    | model  # Obtiene la respuesta del modelo
    | StrOutputParser()  # Parsea la respuesta del modelo
)

In [0]:
# Probemos la cadena RAG con un ejemplo de entrada:
input_example = {"messages": [ {"role": "user", "content":  "¿Qué cubre el plan de beneficios Northwind Health Plus?"}]}
answer = chain.invoke(input_example)
display(answer)

## 📝 Paso 6: Registro del modelo en MLflow
Registramos nuestra chain RAG como modelo de MLflow, incluyendo el LLM endpoint como recurso y los datos necesarios como configuración. ⚠️ Esta celda requiere que el índice FAISS exista y esté accesible en el momento del registro.

In [0]:
def load_retriever(faiss_path):
    # Paso 2: Recargar el índice para compatibilidad con LangChain + MLflow
    embeddings = DatabricksEmbeddings(endpoint="databricks-gte-large-en")
    faiss_index = FAISS.load_local(faiss_path, embeddings,allow_dangerous_deserialization=True)
    retriever = faiss_index.as_retriever(search_kwargs={"k": 3})

    return retriever

In [0]:
def create_chain(model_config, retriever=load_retriever(faiss_path)):
    # Paso 3: Construir la chain
    def extract_user_query(messages):
        return messages[-1]["content"]

    def format_context(docs):
        return "\n".join([f"Passage: {d.page_content}" for d in docs])

    # Plantilla de prompt para el chat
    prompt = ChatPromptTemplate.from_messages(
        [  
            ("system", model_config.get("llm_prompt_template")), # Contiene las instrucciones de la configuración
            ("user", "{question}") # Preguntas del usuario
        ]
    )

    model = ChatDatabricks(
        endpoint=model_config.get("llm_model_serving_endpoint_name"),
        extra_params={"temperature": 0.01, "max_tokens": 500}
    )


    return (
        {
            "question": itemgetter("messages") | RunnableLambda(extract_user_query),
            "context": itemgetter("messages")
            | RunnableLambda(extract_user_query)
            | retriever
            | RunnableLambda(format_context),
        }
        | prompt
        | model
        | StrOutputParser()
    )



In [0]:
# Empezamos el tracking en MLflow
with mlflow.start_run(run_name="log_rag_from_chain"):
    # Crear la chain directamente en memoria usando la configuración proporcionada
    chain = create_chain(chain_config, retriever=load_retriever(faiss_path))
    
    # Registrar el modelo de LangChain en MLflow
    logged_model = mlflow.langchain.log_model(
        lc_model=chain,  # Modelo de LangChain a registrar
        artifact_path="rag_chain",  # Ruta donde se almacenarán los artefactos del modelo
        loader_fn=load_retriever,  # Función para cargar el índice FAISS
        persist_dir=faiss_path,  # Directorio donde se encuentra el índice FAISS
        model_config=chain_config,  # Configuración del modelo
        input_example={"messages": [{"role": "user", "content": "¿Qué cubre Northwind Health Plus?"}]},  # Ejemplo de entrada para el modelo
        example_no_conversion=True,  # Indica que no se debe convertir el ejemplo de entrada
        resources=[
            DatabricksServingEndpoint(endpoint_name=chain_config["llm_model_serving_endpoint_name"])  # Endpoint de servicio de Databricks
        ]
    )

In [0]:
# Define the catalog and database for model registration
catalog = "workspace"
db = "default"
MODEL_NAME = "basic_rag_demo"
MODEL_NAME_FQN = f"{catalog}.{db}.{MODEL_NAME}"

# Register the model to Unity Catalog (UC)
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_model.model_uri,  # URI of the logged model
    name=MODEL_NAME_FQN  # Fully qualified name for the model in UC
)

In [0]:
# Desplegar para habilitar la aplicación de revisión y crear un endpoint de API
# Nota: reducir a cero proporcionará un comportamiento inesperado para la aplicación de chat. Establecer en false para una aplicación lista para producción.
deployment_info = agents.deploy(MODEL_NAME_FQN, model_version=uc_registered_model_info.version, scale_to_zero=True)

instrucciones_para_revisor = f"""## Instrucciones para probar el chatbot asistente de documentación de Databricks

Sus aportes son invaluables para el equipo de desarrollo. Al proporcionar comentarios detallados y correcciones, nos ayuda a solucionar problemas y mejorar la calidad general de la aplicación. Confiamos en su experiencia para identificar cualquier brecha o área que necesite mejora."""

# Agregar las instrucciones para el usuario a la aplicación de revisión
agents.set_review_instructions(MODEL_NAME_FQN, instrucciones_para_revisor)

In [0]:
print(f"\n\nURL de la aplicación de revisión para compartir con sus interesados: {deployment_info.review_app_url}")